# Predicting Used Car Prices with Machine Learning

## 1. Project Objective

The goal of this project is to develop machine learning models capable of predicting used car prices based on vehicle-related features such as mileage, model year, fuel type, transmission type, and manufacturer information.

This project follows a complete data science pipeline including:
- Data loading and inspection
- Data cleaning and preprocessing
- Exploratory Data Analysis (EDA)
- Feature engineering
- Model training
- Model evaluation
- Visualization of results

Two machine learning models will be implemented and compared:
1. Linear Regression
2. Random Forest Regressor

The models will be evaluated using:
- Mean Absolute Error (MAE)
- Root Mean Squared Error (RMSE)
- R² Score

In [2]:
# Import Libraries

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# Visualization style
sns.set_style("whitegrid")

print("Libraries imported successfully.")

Libraries imported successfully.


## 2. Load Dataset

In this section, the raw dataset is loaded into a pandas DataFrame. Initial inspection will help identify:
- Number of rows and columns
- Data types
- Missing values
- Potential preprocessing requirements

In [4]:
# Load Dataset

df = pd.read_csv("data/raw/train.csv")

print("Dataset loaded successfully.")

Dataset loaded successfully.


In [5]:
df.head()

,Unnamed: 0,Name,Location,Year,Kilometers_Driven,Fuel_Type,Transmission,Owner_Type,Mileage,Engine,Power,Seats,New_Price,Price
0,1,Hyundai Creta 1.6 CRDi SX Option,Pune,2015,41000,Diesel,Manual,First,19.67 kmpl,1582 CC,126.2 bhp,5.0,NaN,12.50
1,2,Honda Jazz V,Chennai,2011,46000,Petrol,Manual,First,13 km/kg,1199 CC,88.7 bhp,5.0,8.61 Lakh,4.50
2,3,Maruti Ertiga VDI,Chennai,2012,87000,Diesel,Manual,First,20.77 kmpl,1248 CC,88.76 bhp,7.0,NaN,6.00
3,4,Audi A4 New 2.0 TDI Multitronic,Coimbatore,2013,40670,Diesel,Automatic,Second,15.2 kmpl,1968 CC,140.8 bhp,5.0,NaN,17.74
4,6,Nissan Micra Diesel XV,Jaipur,2013,86999,Diesel,Manual,First,23.08 kmpl,1461 CC,63.1 bhp,5.0,NaN,3.50


## 3. Initial Data Inspection

This section explores the structure of the dataset including:
- Shape of the dataset
- Column names
- Data types
- Missing values
- Basic statistical summaries

In [6]:
# Dataset dimensions
print("Dataset Shape:")
print(df.shape)

Dataset Shape:
(5847, 14)


In [7]:
# Column names
print("Columns:")
print(df.columns)

Columns:
Index(['Unnamed: 0', 'Name', 'Location', 'Year', 'Kilometers_Driven',
       'Fuel_Type', 'Transmission', 'Owner_Type', 'Mileage', 'Engine', 'Power',
       'Seats', 'New_Price', 'Price'],
      dtype='object')


In [8]:
# Dataset information
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5847 entries, 0 to 5846
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Unnamed: 0         5847 non-null   int64  
 1   Name               5847 non-null   object 
 2   Location           5847 non-null   object 
 3   Year               5847 non-null   int64  
 4   Kilometers_Driven  5847 non-null   int64  
 5   Fuel_Type          5847 non-null   object 
 6   Transmission       5847 non-null   object 
 7   Owner_Type         5847 non-null   object 
 8   Mileage            5845 non-null   object 
 9   Engine             5811 non-null   object 
 10  Power              5811 non-null   object 
 11  Seats              5809 non-null   float64
 12  New_Price          815 non-null    object 
 13  Price              5847 non-null   float64
dtypes: float64(2), int64(3), object(9)
memory usage: 639.6+ KB


## 4. Data Cleaning and Preprocessing

This section prepares the dataset for machine learning by:
- Removing unnecessary columns
- Handling missing values
- Separating numerical and categorical features
- Preparing data for model training

Proper preprocessing is important to improve model performance and avoid data leakage.

In [9]:
# Drop Unnecessary Columns

df = df.drop(columns=["Unnamed: 0", "New_Price"])

print("Updated columns:")
print(df.columns)

Updated columns:
Index(['Name', 'Location', 'Year', 'Kilometers_Driven', 'Fuel_Type',
       'Transmission', 'Owner_Type', 'Mileage', 'Engine', 'Power', 'Seats',
       'Price'],
      dtype='object')


In [10]:
# Missing Values

missing_values = df.isnull().sum()

print("Missing values per column:")
print(missing_values)

Missing values per column:
Name                  0
Location              0
Year                  0
Kilometers_Driven     0
Fuel_Type             0
Transmission          0
Owner_Type            0
Mileage               2
Engine               36
Power                36
Seats                38
Price                 0
dtype: int64


### Cleaning Numeric Text Features

Some numerical columns contain units stored as text values:
- Mileage contains "kmpl"
- Engine contains "CC"
- Power contains "bhp"

These columns will be cleaned and converted into numerical format for modeling.

In [11]:
# Clean Mileage Column


df["Mileage"] = df["Mileage"].str.extract(r'(\d+\.?\d*)').astype(float)

print(df["Mileage"].head())

0    19.67
1    13.00
2    20.77
3    15.20
4    23.08
Name: Mileage, dtype: float64


In [12]:
# Clean Engine Column

df["Engine"] = df["Engine"].str.extract(r'(\d+\.?\d*)').astype(float)

print(df["Engine"].head())

0    1582.0
1    1199.0
2    1248.0
3    1968.0
4    1461.0
Name: Engine, dtype: float64


In [13]:
# Clean Power Column

df["Power"] = df["Power"].str.extract(r'(\d+\.?\d*)').astype(float)

print(df["Power"].head())

0    126.20
1     88.70
2     88.76
3    140.80
4     63.10
Name: Power, dtype: float64


In [14]:
# Check updated data types
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5847 entries, 0 to 5846
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Name               5847 non-null   object 
 1   Location           5847 non-null   object 
 2   Year               5847 non-null   int64  
 3   Kilometers_Driven  5847 non-null   int64  
 4   Fuel_Type          5847 non-null   object 
 5   Transmission       5847 non-null   object 
 6   Owner_Type         5847 non-null   object 
 7   Mileage            5845 non-null   float64
 8   Engine             5811 non-null   float64
 9   Power              5811 non-null   float64
 10  Seats              5809 non-null   float64
 11  Price              5847 non-null   float64
dtypes: float64(5), int64(2), object(5)
memory usage: 548.3+ KB
